# 05 — Product Category & Revenue Analysis
**Project:** Olist Brazilian E-Commerce — Customer Behavior & Cohort Analysis

This notebook explores product category performance across revenue, order volume, average order value, and customer satisfaction — and examines payment method preferences and delivery performance by category.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_raw, build_master, build_items_master
from src.utils import apply_style, PALETTE

apply_style()
raw = load_raw()
master = build_master(raw)
items  = build_items_master(raw)
print('Data loaded ✓')

## 5.1 Top Categories by Revenue & Volume

In [ ]:
top_cats = (
    items.groupby('product_category_name_english')
    .agg(revenue=('price','sum'), orders=('order_id','nunique'))
    .assign(aov=lambda df: df['revenue'] / df['orders'])
    .sort_values('revenue', ascending=False)
    .head(15)
    .reset_index()
)
top_cats['category_label'] = top_cats['product_category_name_english'].str.replace('_',' ').str.title()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, col, label, cmap in zip(
    axes,
    ['revenue', 'orders', 'aov'],
    ['Revenue (R$)', 'Order Volume', 'Avg Order Value (R$)'],
    ['Blues_r', 'Purples_r', 'Greens_r']
):
    sorted_df = top_cats.sort_values(col)
    colors = plt.get_cmap(cmap)([i/len(sorted_df) for i in range(len(sorted_df))])
    ax.barh(sorted_df['category_label'], sorted_df[col], color=colors, edgecolor='white')
    ax.set_title(f'Top 15 Categories\nby {label}', fontsize=11)
    ax.set_xlabel(label)

fig.suptitle('Product Category Performance', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 5.2 Category Revenue Table

In [ ]:
top_cats[['category_label','revenue','orders','aov']].rename(columns={
    'category_label':'Category','revenue':'Revenue (R$)','orders':'Orders','aov':'Avg Order Value'
}).style.format({
    'Revenue (R$)': 'R$ {:,.0f}',
    'Orders': '{:,}',
    'Avg Order Value': 'R$ {:.0f}'
}).background_gradient(subset=['Revenue (R$)'], cmap='Greens')

## 5.3 Payment Method Mix

In [ ]:
pay = (
    raw['payments'][raw['payments']['order_id'].isin(master['order_id'])]
    .groupby('payment_type')['payment_value'].sum()
    .reset_index()
    .sort_values('payment_value', ascending=False)
)
pay['payment_type'] = pay['payment_type'].str.replace('_',' ').str.title()
pay['pct'] = pay['payment_value'] / pay['payment_value'].sum() * 100

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2563EB','#7C3AED','#10B981','#F59E0B']
wedges, texts, autotexts = ax.pie(
    pay['payment_value'], labels=pay['payment_type'],
    autopct='%1.1f%%', colors=colors, startangle=140,
    wedgeprops={'edgecolor':'white','linewidth':2}, pctdistance=0.75
)
ax.set_title('Payment Method Distribution (by Revenue)', fontsize=13)
plt.tight_layout()
plt.show()

print(pay.to_string(index=False))

## 5.4 Review Scores vs Delivery Time

In [ ]:
reviews = raw['reviews'][raw['reviews']['order_id'].isin(master['order_id'])]
review_master = master.merge(reviews[['order_id','review_score']], on='order_id', how='inner')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Avg delivery time by review score
avg_del = review_master.groupby('review_score')['delivery_days'].mean()
score_colors = ['#EF4444','#F59E0B','#FBBF24','#34D399','#10B981']
axes[0].bar(avg_del.index, avg_del.values, color=score_colors, edgecolor='white', width=0.65)
for x, y in zip(avg_del.index, avg_del.values):
    axes[0].text(x, y+0.2, f'{y:.1f}d', ha='center', fontsize=10)
axes[0].set_xlabel('Review Score')
axes[0].set_ylabel('Avg Delivery Days')
axes[0].set_title('Avg Delivery Time by Review Score', fontsize=12)

# Score distribution
score_counts = review_master['review_score'].value_counts().sort_index()
axes[1].bar(score_counts.index, score_counts.values, color=score_colors, edgecolor='white', width=0.65)
for x, y in zip(score_counts.index, score_counts.values):
    axes[1].text(x, y+200, f'{y:,}', ha='center', fontsize=9)
axes[1].set_xlabel('Review Score')
axes[1].set_ylabel('Number of Reviews')
axes[1].set_title(f'Review Score Distribution\n(Avg: {review_master["review_score"].mean():.2f}/5.0)', fontsize=12)

fig.suptitle('Customer Satisfaction & Delivery Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Key Takeaways
- **Health & Beauty** leads in revenue; **Watches & Gifts** has the highest AOV (~R$ 212/order)
- **Credit card** dominates at 78.5% of revenue — potential to expand reach with Pix or BNPL integration
- Delivery time is **strongly correlated** with review score — 1-star orders average ~20 days vs ~9 days for 5-star orders
- Reducing late deliveries is the single most actionable lever for improving customer satisfaction

---
**Analysis complete.** See the full shareable report at `outputs/reports/olist_customer_analysis_report.html`